# Phase 5 — Transfer Learning & Pretrained Models

Phase 4's CNN learned entirely from MNIST's 60,000 training images. Most
real projects don't have 60,000 labeled examples for their specific task —
they have hundreds. **Transfer learning** solves this: start from a network
already trained on a huge, general dataset (ImageNet — 1.2 million images,
1000 classes), since its early and middle layers have already learned
general-purpose visual features (edges, textures, shapes) that transfer to
almost any image task.

| Term | Meaning |
|---|---|
| pretrained weights | parameters learned on a large dataset (ImageNet), reused as a starting point |
| backbone | the convolutional layers of a pretrained model, up to (not including) its final classifier |
| freezing | setting `requires_grad=False` so a layer's weights don't change during training |
| feature extraction | freeze the backbone, train only a new classifier head |
| fine-tuning | unfreeze some/all of the backbone too, and keep training (usually at a lower `lr`) |
| differential learning rates | a different `lr` per group of parameters (small for the backbone, larger for the head) |

## Roadmap for this notebook
```
Load a pretrained ResNet18 -> a tiny 2-class dataset (cat vs dog, deliberately small)
    -> baseline: a small CNN trained from scratch on it
    -> feature extraction: freeze ResNet18, train only a new head
    -> fine-tuning: unfreeze everything, differential learning rates
    -> compare trainable params & accuracy across all three
```


## Section 1 — Load a Pretrained Model

`torchvision.models` ships architectures with optional pretrained weights.
`ResNet18_Weights.DEFAULT` is the best available weight set for this
architecture — trained on ImageNet-1k. Crucially, `weights.transforms()`
returns the **exact preprocessing pipeline used while training those
weights** (resize size, crop size, normalization stats) — using it avoids
hand-copying ImageNet mean/std constants and subtly getting them wrong.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

weights = ResNet18_Weights.DEFAULT   # best available weights, trained on ImageNet-1k
resnet = resnet18(weights=weights)

top1 = weights.meta["_metrics"]["ImageNet-1K"]["acc@1"]
print(f"Pretrained on ImageNet-1k -- top-1 accuracy there: {top1}%")
print(f"\nOriginal classifier head: {resnet.fc}   <- 1000 ImageNet classes")

total_params = sum(p.numel() for p in resnet.parameters())
print(f"Total parameters: {total_params:,}")

# ── PREPROCESSING — must match what the model saw during training ────────────
preprocess = weights.transforms()
print(f"\nRequired preprocessing:\n{preprocess}")

## Section 2 — A Deliberately Tiny Dataset

CIFAR-10 has 5,000 training images per class — plenty. To make the case for
transfer learning honestly, we'll only use a small slice of it: **200
images per class for training, 100 per class for testing**. This is closer
to what a real "I have a folder of photos" project looks like, and it's
exactly the regime where a pretrained backbone has the biggest advantage
over training from scratch.

In [ ]:
import numpy as np
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

CLASS_NAMES = ["cat", "dog"]
CIFAR_CAT_IDX, CIFAR_DOG_IDX = 3, 5   # CIFAR-10's own class indices for these two classes

def remap_label(y):
    return 0 if y == CIFAR_CAT_IDX else 1   # cat -> 0, dog -> 1 (CrossEntropyLoss needs 0..num_classes-1)

train_full = CIFAR10(root="data", train=True, download=True,
                      transform=preprocess, target_transform=remap_label)
test_full = CIFAR10(root="data", train=False, download=True,
                     transform=preprocess, target_transform=remap_label)

def class_subset(dataset, n_per_class, seed=42):
    """Indices for exactly n_per_class of each cat/dog example, shuffled."""
    rng = np.random.default_rng(seed)
    targets = np.array(dataset.targets)
    indices = []
    for cifar_idx in (CIFAR_CAT_IDX, CIFAR_DOG_IDX):
        class_indices = np.where(targets == cifar_idx)[0]
        indices.extend(rng.choice(class_indices, size=n_per_class, replace=False))
    rng.shuffle(indices)
    return indices

train_indices = class_subset(train_full, n_per_class=200)
test_indices = class_subset(test_full, n_per_class=100)

train_data = Subset(train_full, train_indices)
test_data = Subset(test_full, test_indices)

print(f"Train samples: {len(train_data)}  ({len(train_data) // 2} per class)")
print(f"Test samples : {len(test_data)}  ({len(test_data) // 2} per class)")

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# ── VISUALIZE A FEW SAMPLES ───────────────────────────────────────────────────
# preprocess normalized these images, so pixel values are no longer in [0, 1] --
# undo that (x * std + mean) before displaying them.
IMAGENET_MEAN = torch.tensor(preprocess.mean).view(3, 1, 1)
IMAGENET_STD = torch.tensor(preprocess.std).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor * IMAGENET_STD + IMAGENET_MEAN
    return img.clamp(0, 1).permute(1, 2, 0).numpy()   # (C,H,W) -> (H,W,C) for imshow

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for i, ax in enumerate(axes):
    image, label = train_data[i]
    ax.imshow(denormalize(image))
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
plt.suptitle("Sample images from the tiny cat/dog subset (denormalized for display)")
plt.show()

## Section 3 — Baseline: a Small CNN Trained From Scratch

A `MiniCNN` in the same style as Phase 4's `SimpleCNN`, but ending in
`nn.AdaptiveAvgPool2d(1)` instead of a hardcoded flatten size — this makes
the classifier head work regardless of the exact input resolution, since
adaptive pooling always collapses whatever spatial size remains down to
`1x1`. Trained on nothing but our 400 tiny training images.

In [ ]:
class MiniCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),   # collapses whatever spatial size remains to 1x1
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # (batch, 64, 1, 1) -> (batch, 64)
        return self.classifier(x)


def train_model(model, loader, optimizer, loss_fn, epochs, label):
    model.train()
    history = []
    for epoch in range(epochs):
        total_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        history.append(avg_loss)
        print(f"[{label}] Epoch {epoch + 1}/{epochs} | Avg Loss: {avg_loss:.4f}")
    return history


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


loss_fn = nn.CrossEntropyLoss()

baseline = MiniCNN().to(device)
baseline_optimizer = optim.Adam(baseline.parameters(), lr=0.001)

baseline_history = train_model(baseline, train_loader, baseline_optimizer, loss_fn,
                                epochs=5, label="baseline")
baseline_acc = evaluate(baseline, test_loader)
baseline_trainable = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
print(f"\nBaseline (from scratch) test accuracy: {baseline_acc:.2f}%")

## Section 4 — Feature Extraction: Freeze the Backbone, Train a New Head

Load a fresh pretrained ResNet18, freeze every one of its parameters, then
replace `fc` (the final layer, originally mapping to 1000 ImageNet classes)
with a brand-new `nn.Linear` for our 2 classes. A newly created layer
defaults to `requires_grad=True`, so it's the only thing that trains.

In [ ]:
feature_extractor = resnet18(weights=weights)

# Freeze every backbone parameter -- we only want to train a brand new head
for param in feature_extractor.parameters():
    param.requires_grad = False

# Replace the final layer: 1000 ImageNet classes -> 2 classes (cat/dog)
feature_extractor.fc = nn.Linear(feature_extractor.fc.in_features, len(CLASS_NAMES))
feature_extractor = feature_extractor.to(device)

fe_trainable = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)
fe_total = sum(p.numel() for p in feature_extractor.parameters())
print(f"Trainable parameters: {fe_trainable:,} / {fe_total:,} total ({100 * fe_trainable / fe_total:.2f}%)")

# filter() picks out only the parameters that still require grad -- here, just resnet.fc
fe_optimizer = optim.Adam(filter(lambda p: p.requires_grad, feature_extractor.parameters()), lr=0.001)

fe_history = train_model(feature_extractor, train_loader, fe_optimizer, loss_fn,
                          epochs=5, label="feature-extraction")
fe_acc = evaluate(feature_extractor, test_loader)
print(f"\nFeature-extraction test accuracy: {fe_acc:.2f}%")

## Section 5 — Fine-Tuning: Unfreeze Everything, Differential Learning Rates

Continuing from the feature-extraction model (its head is already trained),
unfreeze the whole backbone too. The backbone's weights already work well —
they just need small nudges — so it gets a much smaller learning rate than
the head, via **parameter groups**.

In [ ]:
for param in feature_extractor.parameters():
    param.requires_grad = True

ft_trainable = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)
print(f"Trainable parameters: {ft_trainable:,}  (the entire model, now unfrozen)")

# Two parameter groups, two learning rates: small for the pretrained backbone,
# the head's original (larger) rate for the newer, less-trained layer.
ft_optimizer = optim.Adam([
    {"params": feature_extractor.fc.parameters(), "lr": 1e-3},
    {"params": (p for n, p in feature_extractor.named_parameters() if not n.startswith("fc")), "lr": 1e-5},
])

ft_history = train_model(feature_extractor, train_loader, ft_optimizer, loss_fn,
                          epochs=3, label="fine-tune")
ft_acc = evaluate(feature_extractor, test_loader)
print(f"\nFine-tuned test accuracy: {ft_acc:.2f}%")

## Section 6 — Comparing All Three

Same tiny 400-image training set for every model. What differs is how much
of the network is actually learning from that data.

In [ ]:
print(f"{'Model':<28}{'Trainable Params':>18}{'Test Accuracy':>16}")
print(f"{'MiniCNN (from scratch)':<28}{baseline_trainable:>18,}{baseline_acc:>15.2f}%")
print(f"{'ResNet18 (feature extract)':<28}{fe_trainable:>18,}{fe_acc:>15.2f}%")
print(f"{'ResNet18 (fine-tuned)':<28}{ft_trainable:>18,}{ft_acc:>15.2f}%")

plt.plot(range(1, len(baseline_history) + 1), baseline_history, marker="o", label="MiniCNN (from scratch)")
plt.plot(range(1, len(fe_history) + 1), fe_history, marker="o", label="ResNet18 (feature extraction)")
plt.xlabel("Epoch")
plt.ylabel("Average training loss")
plt.title("Training loss: from-scratch CNN vs frozen pretrained ResNet18 (same 5 epochs)")
plt.legend()
plt.grid(True)
plt.show()

## Section 7 — Visualize Predictions

The fine-tuned model's predictions on random test images.

In [ ]:
feature_extractor.eval()
n_show = 8
indices = torch.randint(0, len(test_data), (n_show,))

fig, axes = plt.subplots(1, n_show, figsize=(14, 2.4))
with torch.no_grad():
    for ax, idx in zip(axes, indices):
        image, true_label = test_data[idx]
        input_batch = image.unsqueeze(0).to(device)
        predicted_label = feature_extractor(input_batch).argmax(dim=1).item()

        ax.imshow(denormalize(image))
        is_correct = predicted_label == true_label
        ax.set_title(f"pred {CLASS_NAMES[predicted_label]}\ntrue {CLASS_NAMES[true_label]}",
                     color="green" if is_correct else "red")
        ax.axis("off")

plt.suptitle("Fine-tuned ResNet18 predictions (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

## Phase 5 Summary

```
Loading a pretrained model:
  weights = ResNet18_Weights.DEFAULT
  model = resnet18(weights=weights)
  preprocess = weights.transforms()   -> the EXACT preprocessing used during training

Feature extraction (small dataset, task similar to ImageNet):
  for p in model.parameters(): p.requires_grad = False
  model.fc = nn.Linear(model.fc.in_features, num_classes)   -> new layer, requires_grad=True by default
  optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=...)

Fine-tuning (more data available, or task differs more from ImageNet):
  for p in model.parameters(): p.requires_grad = True
  optim.Adam([
      {"params": model.fc.parameters(), "lr": 1e-3},       # new layer: larger lr
      {"params": backbone_params, "lr": 1e-5},              # pretrained layers: small lr
  ])

Rule of thumb:
  tiny dataset, similar to ImageNet   -> feature extraction (fast, low overfitting risk)
  larger dataset, or quite different  -> fine-tune more layers, smaller backbone lr
  AdaptiveAvgPool2d(1)                -> classifier head works for any input spatial size

Up Next (Phase 6 — Training Techniques):
  data augmentation, Dropout/BatchNorm, LR scheduling, early stopping --
  the techniques that make training actually reliable, not just possible.
```